# House Price Prediction: Linear vs. Ridge Regression

This notebook explores the UCI Real Estate Valuation dataset, visualizes its features, compares Linear Regression and Ridge Regression, and predicts price per unit area for an example property.

> **Interpretation:** The target is price per unit area, not the total price of a home. Dataset values are reported in the source dataset's original units. Treat predictions as educational estimates, not professional valuations.

### Notebook outline
1. Load and inspect the data
2. Explore distributions and feature relationships
3. Split the data and train regression models
4. Evaluate and visualize test-set predictions
5. Run five-fold cross-validation
6. Predict for an example property

In [ ]:
%pip install ucimlrepo pandas numpy matplotlib seaborn scikit-learn


## 1. Load and inspect the dataset

The dataset contains six property/location features and a target value for house price per unit area. We use readable feature names to make later tables and plots easier to understand.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 4775
sns.set_theme(style='whitegrid', context='notebook')

# Fetch dataset 477 from the UCI Machine Learning Repository.
real_estate = fetch_ucirepo(id=477)
X = real_estate.data.features.copy()
y = real_estate.data.targets.iloc[:, 0].copy()

X.columns = [
    'transaction_date',
    'house_age',
    'distance_mrt',
    'num_convenience_stores',
    'latitude',
    'longitude',
]
y.name = 'house_price_unit_area'
df = X.join(y)

print(f'Dataset rows: {len(df)} | Features: {X.shape[1]}')
display(df.head())

In [ ]:
# Check data types, summary statistics, and missing values before modeling.
print('Data types and non-null counts:')
df.info()
print('\nSummary statistics:')
display(df.describe().T.round(2))
print('\nMissing values by column:')
display(df.isna().sum().to_frame('missing values'))

## 2. Explore distributions and relationships

The plots below help reveal the target's distribution and how each feature is associated with price. Correlation is descriptive and does not establish cause and effect.

In [ ]:
# Plot the target distribution and pairwise feature/target relationships.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['house_price_unit_area'], kde=True, ax=axes[0], color='#3973ac')
axes[0].set(title='Distribution of price per unit area', xlabel='Price per unit area', ylabel='Number of properties')
sns.scatterplot(data=df, x='distance_mrt', y='house_price_unit_area', alpha=0.65, ax=axes[1], color='#d17a22')
axes[1].set(title='Price vs. distance to nearest MRT station', xlabel='Distance to MRT station', ylabel='Price per unit area')
plt.tight_layout()
plt.show()

# Correlation heatmap summarizes linear relationships among numeric columns.
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='vlag', center=0, linewidths=0.5)
plt.title('Feature and target correlation matrix')
plt.tight_layout()
plt.show()

## 3. Split data and train models

A held-out test set is reserved for final evaluation. Imputation and standardization are placed inside each pipeline, so they are learned from training data only. Standardization is especially useful for Ridge Regression because its penalty depends on coefficient scale.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)

def make_pipeline(regressor):
    # Fit preprocessing only on each training fold to avoid leakage.
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('regressor', regressor),
    ])

models = {
    'Linear Regression': make_pipeline(LinearRegression()),
    'Ridge Regression': make_pipeline(Ridge(alpha=1.0)),
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)

print(f'Training rows: {len(X_train)} | Test rows: {len(X_test)}')
print('Both models trained successfully.')

In [ ]:
# Compare models using the same held-out observations.
# MAE and RMSE are in target units; larger R² indicates a closer fit.
metric_rows = []
for name, predicted in predictions.items():
    metric_rows.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predicted),
        'RMSE': mean_squared_error(y_test, predicted) ** 0.5,
        'R²': r2_score(y_test, predicted),
    })
metrics = pd.DataFrame(metric_rows).set_index('Model')
display(metrics.round(3))

## 4. Visualize test-set predictions and errors

In the actual-versus-predicted plot, points near the dashed line are more accurate. The residual plot helps identify systematic errors; a residual is actual minus predicted.

In [ ]:
# Show actual-versus-predicted and residual plots for both models.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Linear Regression': '#3973ac', 'Ridge Regression': '#d17a22'}
all_values = np.concatenate([y_test.to_numpy()] + [p for p in predictions.values()])
low, high = all_values.min(), all_values.max()

for name, predicted in predictions.items():
    axes[0].scatter(y_test, predicted, alpha=0.65, s=35, label=name, color=colors[name])
    residuals = y_test.to_numpy() - predicted
    axes[1].scatter(predicted, residuals, alpha=0.65, s=35, label=name, color=colors[name])

axes[0].plot([low, high], [low, high], 'k--', linewidth=1.5, label='Ideal prediction')
axes[0].set(title='Actual vs. predicted (test set)', xlabel='Actual price per unit area', ylabel='Predicted price per unit area')
axes[0].legend()
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set(title='Residuals (test set)', xlabel='Predicted price per unit area', ylabel='Actual − predicted')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Five-fold cross-validation

Cross-validation repeats training and evaluation across five data folds. The mean and standard deviation of R² provide a view of performance stability beyond one train/test split.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='r2')
    cv_rows.append({
        'Model': name,
        'Fold R² scores': np.round(scores, 3).tolist(),
        'Mean R²': scores.mean(),
        'Std. dev.': scores.std(),
    })
cv_results = pd.DataFrame(cv_rows).set_index('Model')
display(cv_results.round(3))

## 6. Predict for an example property

The example below is illustrative. For a useful real-world estimate, provide values in the same units and ranges as the training data, and validate the model against current local market data.

In [ ]:
example_house = pd.DataFrame([{
    'transaction_date': 2013.25,
    'house_age': 20.0,
    'distance_mrt': 500.0,
    'num_convenience_stores': 5,
    'latitude': 24.98,
    'longitude': 121.54,
}])

# Use Ridge Regression for this example; change the key to use Linear Regression.
chosen_model = models['Ridge Regression']
estimated_price = chosen_model.predict(example_house)[0]
print(f'Estimated price per unit area: {estimated_price:.2f}')
print('This is an educational estimate, not a guaranteed sale price.')

## Key takeaways

- Compare MAE, RMSE, and R² together; no single metric tells the whole story.
- Use the residual plot to look for systematic over- or under-prediction.
- Cross-validation provides a more stable performance estimate than a single split.
- The dataset is limited and historical; results should not be treated as current market guidance.